# Lab 6.1 &mdash; A Retriever You Can Inspect

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Chunk a document two ways, and watch one of them make an answer unreachable
- Score, rank, and see that top-k always returns k &mdash; whatever is in the corpus
- Build the score floor that turns &lsquo;the least bad thing&rsquo; into an empty result
- Scope with metadata, which is what production retrieval actually looks like

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **No embeddings here, deliberately.** The scoring function is a stand-in so every
> check is exact and offline. Chunking, ranking, floors and filters are the same
> whatever computes the similarity &mdash; and they decide more than the model does.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents. Read 3.2: the rule and the exception that qualifies it are
# adjacent sentences, which is the whole of Lab 6.1's first lesson. Note also what is NOT
# here -- there is nothing about FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """
## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """
## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

## Concept

A retriever is four decisions, and only one of them is the embedding model:

1. **How the corpus is cut up.** Decides what can ever be returned together.
2. **How a chunk is scored** against a query. This is the part people think is the whole thing.
3. **How many come back, and how bad they are allowed to be.**
4. **What is in scope** before ranking starts.

This lab builds 1, 3 and 4 exactly, and 2 approximately &mdash; because the approximate version is
enough to see everything that matters, and it makes every check deterministic.

## Section 1 &mdash; Chunking decides what can be found

Section 3.2 states a rule and then exempts intra-group transfers from it. Cut that in half and no
retriever can ever return the two together, because they are no longer one thing.

In [ ]:
import re

SECTION_RE = re.compile(r"^##\s+(.*)$", re.M)

def chunk_by_chars(source, text, size=120):
    """The naive chunker: cut every `size` characters, meaning be damned."""
    flat = " ".join(text.split())
    return [{"source": source, "section": f"chars {i}-{i + size}", "text": flat[i:i + size]}
            for i in range(0, len(flat), size)]


def chunk_by_section(source, text):
    """One chunk per section, so a rule and the exception that qualifies it stay together."""
    out, parts = [], SECTION_RE.split(text)
    # parts is [preamble, heading, body, heading, body, ...]
    for i in range(1, len(parts) - 1, 2):
        heading, body = parts[i].strip(), " ".join(parts[i + 1].split())
        out.append({"source": source, "section": heading, "text": heading + " -- " + body})
    return out


def build_index(chunker):
    """Run one chunker over every document."""
    return [c for source, text in DOCS.items() for c in chunker(source, text)]

In [ ]:
# --- Self-check: Section 1
def by_section():
    return build_index(chunk_by_section)

def by_chars():
    return build_index(chunk_by_chars)

def limit_chunk(index):
    """The chunk that states the USD 500,000 rule."""
    return next(c for c in index if "500,000" in c["text"])

check("section chunking finds all six sections across the two documents",
      lambda: len(by_section()) == 6)
check("every chunk knows where it came from",
      lambda: all(c["source"] in DOCS and c["section"] for c in by_section()))
check("the heading is part of what gets scored, not just a label",
      lambda: "Limit breaches" in limit_chunk(by_section())["text"],
      "a query says 'limit breach'; if that phrase is only in the label it cannot be matched")
check("the rule and the exception that qualifies it are in ONE chunk",
      lambda: "intra-group" in limit_chunk(by_section())["text"])
check("character chunking splits them apart",
      lambda: "intra-group" not in limit_chunk(by_chars())["text"],
      "after this cut, no retriever on earth can return them together")
check("and that is not a small-chunk problem -- it is a boundary problem",
      lambda: any("intra-group" in c["text"] for c in by_chars()),
      "the exception is still in the index; it is just no longer attached to the rule it qualifies")

def _show():
    print("  by section:", limit_chunk(by_section())["text"][:96], "...")
    print("  by chars  :", limit_chunk(by_chars())["text"][:96], "...")
guard(_show)

## Section 2 &mdash; Rank, and then refuse to

The scoring function below is term overlap. An embedding would score differently and better; it
would not change anything else in this lab, which is the point.

The important part is the last argument: **top-k always returns k**, so the floor is the only thing
standing between you and four confident irrelevant chunks.

In [ ]:
STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def terms(text):
    return {w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1}


def similarity(query, chunk):
    """How well one chunk answers one query, 0.0 to 1.0. A stand-in for a cosine similarity.

    Named `similarity`, not `score` -- this notebook already has a score() that prints your
    marks, and shadowing it would break the last cell of the lab.
    """
    q = terms(query)
    if not q:
        return 0.0
    return len(q & terms(chunk["text"])) / len(q)


def search(query, index, k=4, floor=0.0):
    """Top-k by score, then drop anything that did not clear the floor."""
    scored = sorted(((similarity(query, c), c) for c in index), key=lambda sc: -sc[0])
    top = scored[:k]
    return [{"score": round(s, 3), **c} for s, c in top if s >= floor]

In [ ]:
# --- Self-check: Section 2
LIMIT_Q = "what approval does a limit breach above 500,000 need"
JPY_Q   = "what is the FX hedging policy for JPY exposure"

check("a real question finds the right section first",
      lambda: search(LIMIT_Q, by_section())[0]["section"].startswith("3.2"))
check("results come back ranked, best first",
      lambda: [r["score"] for r in search(LIMIT_Q, by_section())]
              == sorted((r["score"] for r in search(LIMIT_Q, by_section())), reverse=True))
check("k is respected",
      lambda: len(search(LIMIT_Q, by_section(), k=2)) <= 2)
check("A QUESTION THE CORPUS CANNOT ANSWER STILL RETURNS FOUR RESULTS",
      lambda: len(search(JPY_Q, by_section(), k=4)) == 4,
      "nothing in either document mentions FX or JPY, and four chunks come back anyway")
check("scoring zero is not the same as being excluded",
      lambda: all(r["score"] == 0.0 for r in search(JPY_Q, by_section())),
      "a real embedding never returns exactly zero either -- it returns a small number, and k rows")
check("and none of them is about FX",
      lambda: not any("hedg" in r["text"].lower() for r in search(JPY_Q, by_section())))
check("a floor turns that into an empty result",
      lambda: search(JPY_Q, by_section(), floor=0.25) == [],
      "this is the only thing that lets the agent say 'I could not find it'")
check("and the same floor does not break the good query",
      lambda: len(search(LIMIT_Q, by_section(), floor=0.25)) > 0)
check("the floor has to be chosen against the corpus, not guessed",
      lambda: search(LIMIT_Q, by_section(), floor=0.95) == [],
      "set it too high and every question refuses -- Lab 6.5 measures where it should sit")

def _ranked():
    for q, label in ((LIMIT_Q, "answerable"), (JPY_Q, "not in the corpus")):
        print(f"  [{label}] {q}")
        for r in search(q, by_section()):
            print(f"      {r['score']:.2f}  {r['source']:24} {r['section']}")
        print()
guard(_ranked)

## Section 3 &mdash; Scope before you rank

Most production retrieval is a metadata filter with a similarity search inside it: this version,
this jurisdiction, documents this user is allowed to see. An unfiltered index is a disclosure
waiting to be reported.

In [ ]:
def search_scoped(query, index, k=4, floor=0.0, where=None):
    """Narrow by metadata first, then rank inside the scope."""
    where = where or {}
    scope = [c for c in index if all(c.get(key) == val for key, val in where.items())]
    return search(query, scope, k=k, floor=floor)

In [ ]:
# --- Self-check: Section 3
check("no filter searches everything",
      lambda: len(search_scoped(LIMIT_Q, by_section()))
              == len(search(LIMIT_Q, by_section())))
check("a source filter restricts the results to that document",
      lambda: all(r["source"] == "escalation-policy-v2.md"
                  for r in search_scoped("who may approve a release", by_section(),
                                         where={"source": "escalation-policy-v2.md"})))
check("and it changes the answer, which is the whole point",
      lambda: search_scoped("who may approve a release", by_section(),
                            where={"source": "escalation-policy-v2.md"})[0]["section"]
              .startswith("1"))
check("filtering to something that does not exist returns nothing, not everything",
      lambda: search_scoped(LIMIT_Q, by_section(), where={"source": "no-such-doc.md"}) == [],
      "a filter that silently falls back to the whole index is how a disclosure happens")
check("every key in the filter has to match, not just one",
      lambda: search_scoped(LIMIT_Q, by_section(),
                            where={"source": "ops-runbook-v4.md",
                                   "section": "no such section"}) == [])

## Run it for real &mdash; with actual embeddings

Everything above used term overlap. This cell puts the same corpus into ChromaDB with a real
embedding model and asks the same two questions, so you can see where semantics beat words.

**First run downloads about 80 MB** of embedding model, so give it a minute. If it is unavailable
the cell says so and the lab still scores.

In [ ]:
def with_real_embeddings():
    import chromadb
    index = build_index(chunk_by_section)
    client = chromadb.Client()
    col = client.get_or_create_collection("m6-lab1")
    col.add(ids=[f"c{i}" for i in range(len(index))],
            documents=[c["text"] for c in index],
            metadatas=[{"source": c["source"], "section": c["section"]} for c in index])
    for q in (LIMIT_Q, JPY_Q, "can we push a large payment between our own entities"):
        res = col.query(query_texts=[q], n_results=2)
        print(f"  {q}")
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
            print(f"      dist {dist:.3f}  {meta['section']:26} {doc[:52]}")
        print()

try:
    guard(with_real_embeddings)
except Exception as exc:
    print(f"(embeddings unavailable here: {type(exc).__name__}: {exc})")
    print("The graded cells above do not need them.")

### Read it

Look at the third question &mdash; *&ldquo;can we push a large payment between our own entities&rdquo;* &mdash; which
shares almost no words with section 3.2 but means exactly it. Measured on this corpus:

| | top result | is it right? |
|---|---|---|
| term overlap (this lab) | 3.1, 3.3 and 3.4, tied at 0.167 | no &mdash; 3.2 is not even in the top three |
| embeddings (the cell above) | 3.2 Limit breaches, distance 1.073 | yes |

The lexical retriever does not merely score it weakly. It returns three wrong sections, confidently
tied, and the right one never appears. **That** is what the embedding model buys you, and it is
worth having &mdash; nothing in this lab argues otherwise.

Now look at the second question, the one about FX, where the corpus genuinely has nothing.
Embeddings return 3.2 at distance 1.446: further away, still first, still four rows, still no
empty result. The scoring function changed. What did not change is that top-k returns k, that a
badly cut chunk cannot be reassembled, or that an unfiltered index returns things the reader
should not see. Those are the parts you build, and they are the rest of this module.

In [ ]:
score()

## Your turn

1. `chunk_by_chars` splits mid-sentence. Add a 40-character overlap and re-run the Section 1
   checks. Does overlap actually reattach the exception to its rule, or does it just make the
   failure less frequent and harder to find?
2. Sections are uneven &mdash; 3.4 is two sentences, 3.2 is four. Find the section that is too big to
   be a single retrievable idea and split it on meaning. What did you have to decide?
3. Set `floor` to each of 0.1, 0.25 and 0.4 and record, for both questions, whether you got an
   answer and whether it was right. That table is the beginning of Lab 6.5.